# <i><u>Animation notebook</u></i>

This notebook provides the code to generate animations for illustrating the paper <i>From phase space to Krylov space, one shell at a time</i>. These animations are used in the <code>README.md</code> file. We begin this notebook by importing the necessary packages and modules.

In [1]:
import numpy as np
import matplotlib.animation as animation
import matplotlib.pyplot as plt

from KrylovQuantumClassical import classicalLMG, classicalLMG_MC, LanczosQuantum

## <span style="color:orange">1.  Classical and quantum Lanczos coefficients animation</span>

We first compute the classical Lanczos coefficients for the LMG model. We then compute the corresponding quantum Lanczos coefficients for various increasing system sizes.

In [2]:
h = 0.5
J = 1.0
ic_z = [[[1, 0], 1.]]
LMG_classical = classicalLMG(h, J, ic_z)
spectral_width = LMG_classical.spectral_width()

n_max = 100
classical_Lanczos_LMG = LMG_classical.Lanczos_coeff_IT(n_max)

In [3]:
LMG_quantum_5 = LanczosQuantum(model = 'LMG', spin_size = 5, param = [h, J], initial_operator = [[1, 3, 1]], precision = 250)
LMG_quantum_10 = LanczosQuantum(model = 'LMG', spin_size = 10, param = [h, J], initial_operator = [[1, 3, 1]], precision = 350)
LMG_quantum_15 = LanczosQuantum(model = 'LMG', spin_size = 15, param = [h, J], initial_operator = [[1, 3, 1]], precision = 500)
LMG_quantum_20 = LanczosQuantum(model = 'LMG', spin_size = 20, param = [h, J], initial_operator = [[1, 3, 1]], precision = 100)
LMG_quantum_25 = LanczosQuantum(model = 'LMG', spin_size = 25, param = [h, J], initial_operator = [[1, 3, 1]], precision = 1200)
LMG_quantum_30 = LanczosQuantum(model = 'LMG', spin_size = 30, param = [h, J], initial_operator = [[1, 3, 1]], precision = 1500)

LMG_quantum_5.Lanczos_coeff_IT()
LMG_quantum_10.Lanczos_coeff_IT()
LMG_quantum_15.Lanczos_coeff_IT()
LMG_quantum_20.Lanczos_coeff_IT()
LMG_quantum_25.Lanczos_coeff_IT()
LMG_quantum_30.Lanczos_coeff_IT()

LMG_quantum_5.terminate()
LMG_quantum_10.terminate()
LMG_quantum_15.terminate()
LMG_quantum_20.terminate()
LMG_quantum_25.terminate()
LMG_quantum_30.terminate()

We first create the backbone of the animation, which is a static plot containing the classical Lanczos coefficients.

In [4]:
plt.rcParams.update({
    "font.family": "serif",
    "text.usetex": True,
}) # Font settings for LaTeX rendering

fig, ax = plt.subplots(figsize=(9, 6)) # Animation backbone

ax.set_title(r'\textbf{Classical and quantum Lanczos coefficients for the LMG model}', fontsize=15)
ax.set_xlabel(r'$n$', fontsize=17.5)
ax.set_ylabel(r'$b_n/\Lambda(h,J)$', fontsize=17.5)
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize=14, grid_color='grey', grid_alpha=0.5)
ax.set_xlim(0., n_max)
ax.set_ylim(0, 25)

n_axis = np.arange(1, n_max + 1)
ax.plot(n_axis, classical_Lanczos_LMG / spectral_width, color='black', label='Classical LMG') # Static classical plot

plt.close()

We now organize the data into a dictionary for easier access during the animation and create relevant structures for the animation.

In [5]:
data_dict = {
    0: classical_Lanczos_LMG[: n_max] / spectral_width,
    5: LMG_quantum_5.Lanczos[: n_max] / spectral_width,
    10: LMG_quantum_10.Lanczos[: n_max] / spectral_width,
    15: LMG_quantum_15.Lanczos[: n_max] / spectral_width,
    20: LMG_quantum_20.Lanczos[: n_max] / spectral_width,
    25: LMG_quantum_25.Lanczos[: n_max] / spectral_width,
    30: LMG_quantum_30.Lanczos[: n_max] / spectral_width,
}

lines = {}
dots = {} 
configs = [
    (5, 'cornflowerblue'), (10, 'lightcoral'), (15, 'mediumseagreen'),
    (20, 'darkorange'), (25, 'darkviolet'), (30, 'darkgray')]

cmap = plt.colormaps['viridis']
colors = [cmap(i / (len(configs) - 1)) for i in range(len(configs))]

for idx, (S, _) in enumerate(configs):
    # Main line
    line, = ax.plot([], [], color=colors[idx], label=f'$S={S}$')
    lines[S] = line
    
    # Trailing dot (using a circle marker 'o')
    dot, = ax.plot([], [], color=colors[idx], marker='o', markersize=5)
    dots[S] = dot

The following fuctions are used to update the animation frame by frame.

In [6]:
def init():
    """
    This function initializes the lines and dots for the animation. It sets the data for each line and dot to empty lists, effectively clearing them at the start of the animation.

    Returns
    -------
    list
        A list containing all the line and dot objects that will be animated. This is necessary for the FuncAnimation to know which artists to update during the animation.
    """
    for line in lines.values():
        line.set_data([], [])
    for dot in dots.values():
        dot.set_data([], [])

    return list(lines.values()) + list(dots.values())

def update(frame):
    """
    This function is called at each frame of the animation. It updates the data for each line and dot based on the current frame number. The lines represent the Lanczos coefficients for different spin sizes, and the dots represent the last point of each line.

    Parameters
    ----------
    frame : int
        The current frame number in the animation.

    Returns
    -------
    list
        A list containing all the line and dot objects that have been updated. This is necessary for the FuncAnimation to know which artists to redraw during the animation.

    Example
    -------
    (See below for an example of usage)
    """
    current_idx = frame + 1 
    
    for S, line in lines.items():
        if S == 5 and current_idx >= LMG_quantum_5.K_dim: # The case S = 5 terminates before n = 100
            idx = LMG_quantum_5.K_dim - 1
        else:
            idx = current_idx
            
        x_data = n_axis[:idx]
        y_data = (S * data_dict[S][:idx]) / spectral_width
        
        line.set_data(x_data, y_data)
        
        if len(x_data) > 0:
            dots[S].set_data([x_data[-1]], [y_data[-1]])
        else:
            dots[S].set_data([], [])
        
    return list(lines.values()) + list(dots.values())

We finally create the animation using the `FuncAnimation` class from `matplotlib.animation`. 

In [13]:
ax.legend(fontsize=12, loc='upper left')

ani = animation.FuncAnimation(fig, update, frames=n_max, init_func=init, blit=True, interval=75)
plt.show()

ani.save('Animations/Lanczos_animation.gif', writer='imagemagick')

MovieWriter imagemagick unavailable; using Pillow instead.


## <span style="color:orange">2.  Time evolution of microcanonical K-complexity animation</span>

We first compute the microcanonical Lanczos coefficients for the LMG model in the saddle regime within the energy shell containing the saddle point and in another shell away from it. We then compute the corresponding time evolution of K-complexity within each shells.

In [8]:
h = 0.5
J = 1.

E = - 0.5
delta_E = 0.1

LMG_quantum_MC_1 = LanczosQuantum(model = 'LMG', spin_size = 25, param = [h, J], initial_operator = [[1, 3, 1]], precision = 1500)
LMG_quantum_MC_1.Lanczos_coeff_MC(E, delta_E, one_point = True)
LMG_quantum_MC_1.terminate()

E = 0.2

LMG_quantum_MC_2 = LanczosQuantum(model = 'LMG', spin_size = 25, param = [h, J], initial_operator = [[1, 3, 1]], precision = 1500)
LMG_quantum_MC_2.Lanczos_coeff_MC(E, delta_E, one_point = True)
LMG_quantum_MC_2.terminate()    
time_grid_saddle, K_complexity_saddle = LMG_quantum_MC_1.K_complexity(10 ** 5, 1, True)
time_grid_nosaddle, K_complexity_nosaddle = LMG_quantum_MC_2.K_complexity(10 ** 5, 1, True)

LT_K_complexity_saddle, Q_0n_saddle = LMG_quantum_MC_1.LT_K_complexity()
LT_K_complexity_nosaddle, Q_0n_nosaddle = LMG_quantum_MC_2.LT_K_complexity()

As before, we first create the backbone of the animation.

In [9]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.set_title(r'\textbf{Time-evolution of microcanonical K-complexity in the LMG model}', fontsize=15)
ax.set_xlabel(r'$t$', fontsize=17.5)
ax.set_ylabel(r'$C_K(t)$', fontsize=17.5)

ax.xaxis.get_major_formatter().set_powerlimits((0, 0))
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize=14, grid_color='grey', grid_alpha=0.5)

total_frames = len(time_grid_saddle) 
max_time = max(time_grid_saddle[-1], time_grid_nosaddle[-1])
max_complexity = max(np.max(K_complexity_saddle), np.max(K_complexity_nosaddle))

ax.set_xlim(0, max_time)
ax.set_ylim(0, max_complexity * 1.1)

line_saddle, = ax.plot([], [], color=colors[1], lw=2, label=r'$E = -0.5\pm 0.05$')
dot_saddle, = ax.plot([], [], color=colors[1], marker='o', markersize=6)

line_nosaddle, = ax.plot([], [], color=colors[3], lw=2, label=r'$E = 0.2\pm 0.05$')
dot_nosaddle, = ax.plot([], [], color=colors[3], marker='o', markersize=6)

plt.close()

The following fuctions are used to update the animation frame by frame.

In [10]:
def init():
    """
    This function initializes the lines and dots for the animation. It sets the data for each line and dot to empty lists, effectively clearing them at the start of the animation.

    Returns
    -------
    tuple
        A tuple containing all the line and dot objects that will be animated. This is necessary for the FuncAnimation to know which artists to update during the animation.
    """
    line_saddle.set_data([], [])
    dot_saddle.set_data([], [])
    line_nosaddle.set_data([], [])
    dot_nosaddle.set_data([], [])

    return line_saddle, dot_saddle, line_nosaddle, dot_nosaddle

def update(frame):
    """
    This function is called at each frame of the animation. It updates the data for each line and dot based on the current frame number. The lines represent the time evolved K-complexity, and the dots represent the last point of each line.

    Parameters
    ----------
    frame : int
        The current frame number in the animation.

    Returns
    -------
    tuple
        A tuple containing all the line and dot objects that have been updated. This is necessary for the FuncAnimation to know which artists to redraw during the animation.

    Example
    -------
    (See below for an example of usage)
    """
    current_idx = frame + 1
    
    x_sad = time_grid_saddle[:current_idx]
    y_sad = K_complexity_saddle[:current_idx]
    line_saddle.set_data(x_sad, y_sad)
    if len(x_sad) > 0:
        dot_saddle.set_data([x_sad[-1]], [y_sad[-1]])
        
    x_nosad = time_grid_nosaddle[:current_idx]
    y_nosad = K_complexity_nosaddle[:current_idx]
    line_nosaddle.set_data(x_nosad, y_nosad)
    if len(x_nosad) > 0:
        dot_nosaddle.set_data([x_nosad[-1]], [y_nosad[-1]])
        
    return line_saddle, dot_saddle, line_nosaddle, dot_nosaddle

We finally create the animation.

In [17]:
ax.legend(fontsize=12, loc='upper right')

frame_steps = range(0, total_frames, max(1, total_frames // 200)) # Targets ~200 total frames

ani = animation.FuncAnimation(fig, update, frames=frame_steps, init_func=init, blit=True, interval=30)
plt.show()

ani.save('Animations/K_complexity_evolution.gif', writer='imagemagick')

MovieWriter imagemagick unavailable; using Pillow instead.
